# Fashion-MNIST Classification: CNN vs CNN with Batch Normalization

**Course:** IBM AI Engineering Professional Certificate  
**Module:** Deep Neural Networks with PyTorch  
**Author:** Jack Pumpuni Frimpong-Manso  
**Dataset:** [Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist)  
**Estimated Time:** 30 minutes

---

## Learning Objectives
After completing this lab you will be able to:
- Build and train a Convolutional Neural Network (CNN) for multi-class image classification
- Implement Batch Normalization and explain its effect on training stability
- Compare training dynamics (loss, accuracy) of CNN vs CNN_batch
- Evaluate model performance on the Fashion-MNIST validation set

---

## Table of Contents
1. [Environment Setup & Dataset](#1.-Environment-Setup-&-Dataset)
2. [Model Architectures](#2.-Model-Architectures)
3. [Training Both Models](#3.-Training-Both-Models)
4. [Results & Comparison](#4.-Results-&-Comparison)
5. [Error Analysis](#5.-Error-Analysis)
6. [Summary](#6.-Summary)

In [ ]:
%%time
%pip install pandas numpy matplotlib --quiet
%pip install torch==2.8.0+cpu torchvision==0.23.0+cpu torchaudio==2.8.0+cpu \
    --index-url https://download.pytorch.org/whl/cpu --quiet

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dsets
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from PIL import Image
import time

torch.manual_seed(0)
print(f"PyTorch version: {torch.__version__}")
device = torch.device('cpu')
print(f"Device: {device}")

---
## 1. Environment Setup & Dataset

### Fashion-MNIST Overview
- **60,000 training** + **10,000 test** grayscale images
- **28×28 pixels**, 10 clothing categories
- Direct drop-in replacement for MNIST, but harder

| Label | Class |
|---|---|
| 0 | T-shirt/top |
| 1 | Trouser |
| 2 | Pullover |
| 3 | Dress |
| 4 | Coat |
| 5 | Sandal |
| 6 | Shirt |
| 7 | Sneaker |
| 8 | Bag |
| 9 | Ankle boot |

In [ ]:
# Class names for Fashion-MNIST
CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]
IMAGE_SIZE = 16  # Resize 28×28 → 16×16 for faster training

# Compose transforms: Resize + ToTensor
composed = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])

# Download Fashion-MNIST
dataset_train = dsets.FashionMNIST(
    root='.fashion/data', train=True,  transform=composed, download=True
)
dataset_val = dsets.FashionMNIST(
    root='.fashion/data', train=False, transform=composed, download=True
)

print(f"Training samples:   {len(dataset_train):,}")
print(f"Validation samples: {len(dataset_val):,}")
print(f"Image size after resize: {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"Image tensor shape: {dataset_train[0][0].shape}")

In [ ]:
# Show function
def show_data(data_sample, ax=None):
    img, label = data_sample
    if ax is None:
        fig, ax = plt.subplots()
    ax.imshow(img.numpy().reshape(IMAGE_SIZE, IMAGE_SIZE), cmap='gray')
    ax.set_title(f'{CLASS_NAMES[label]}\n(label={label})', fontsize=10)
    ax.axis('off')

# Display first 9 validation samples in a grid
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for i, ax in enumerate(axes.flatten()):
    show_data(dataset_val[i], ax=ax)
plt.suptitle('Fashion-MNIST Validation Samples (resized to 16×16)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fashion_mnist_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fashion_mnist_samples.png")

In [ ]:
# DataLoaders
BATCH_SIZE = 100
train_loader = DataLoader(dataset=dataset_train, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(dataset=dataset_val,   batch_size=BATCH_SIZE, shuffle=False)
print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

---
## 2. Model Architectures

### Architecture overview (16×16 input)

```
Input (1×16×16)
    ↓
Conv2d(1→16, k=5, p=2)  → 16×16×16  [same padding]
[BatchNorm2d(16)]         → optional
ReLU
MaxPool2d(k=2)           → 8×8×16
    ↓
Conv2d(16→32, k=5, p=2) → 8×8×32   [same padding]
[BatchNorm2d(32)]         → optional
ReLU
MaxPool2d(k=2)           → 4×4×32
    ↓
Flatten                  → 32×4×4 = 512
Linear(512 → 10)
[BatchNorm1d(10)]         → optional
    ↓
Output logits (10 classes)
```

In [ ]:
# -----------------------------------------------------------------------
# Standard CNN
# -----------------------------------------------------------------------
class CNN(nn.Module):
    """Two-layer CNN for Fashion-MNIST classification."""

    def __init__(self, out_1=16, out_2=32, number_of_classes=10):
        super(CNN, self).__init__()
        # Block 1
        self.cnn1     = nn.Conv2d(in_channels=1, out_channels=out_1, kernel_size=5, padding=2)
        self.maxpool1 = nn.MaxPool2d(kernel_size=2)
        # Block 2
        self.cnn2     = nn.Conv2d(in_channels=out_1, out_channels=out_2, kernel_size=5,
                                   stride=1, padding=2)
        self.maxpool2 = nn.MaxPool2d(kernel_size=2)
        # Classifier
        self.fc1      = nn.Linear(out_2 * 4 * 4, number_of_classes)

    def forward(self, x):
        x = torch.relu(self.cnn1(x))
        x = self.maxpool1(x)
        x = torch.relu(self.cnn2(x))
        x = self.maxpool2(x)
        x = x.view(x.size(0), -1)   # flatten
        x = self.fc1(x)
        return x


# -----------------------------------------------------------------------
# CNN with Batch Normalization
# -----------------------------------------------------------------------
class CNN_batch(nn.Module):
    """
    Two-layer CNN with Batch Normalization after each conv and the FC layer.

    Benefits of BatchNorm:
    - Reduces internal covariate shift
    - Allows higher learning rates
    - Acts as regulariser (reduces need for dropout)
    - Faster convergence
    """

    def __init__(self, out_1=16, out_2=32, number_of_classes=10):
        super(CNN_batch, self).__init__()
        # Block 1
        self.cnn1      = nn.Conv2d(in_channels=1, out_channels=out_1, kernel_size=5, padding=2)
        self.conv1_bn  = nn.BatchNorm2d(out_1)
        self.maxpool1  = nn.MaxPool2d(kernel_size=2)
        # Block 2
        self.cnn2      = nn.Conv2d(in_channels=out_1, out_channels=out_2, kernel_size=5,
                                    stride=1, padding=2)
        self.conv2_bn  = nn.BatchNorm2d(out_2)
        self.maxpool2  = nn.MaxPool2d(kernel_size=2)
        # Classifier
        self.fc1       = nn.Linear(out_2 * 4 * 4, number_of_classes)
        self.bn_fc1    = nn.BatchNorm1d(number_of_classes)

    def forward(self, x):
        x = torch.relu(self.conv1_bn(self.cnn1(x)))
        x = self.maxpool1(x)
        x = torch.relu(self.conv2_bn(self.cnn2(x)))
        x = self.maxpool2(x)
        x = x.view(x.size(0), -1)
        x = self.bn_fc1(self.fc1(x))
        return x


# Instantiate
torch.manual_seed(0)
model_cnn   = CNN(out_1=16, out_2=32, number_of_classes=10)
model_batch = CNN_batch(out_1=16, out_2=32, number_of_classes=10)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"CNN parameter count:       {count_params(model_cnn):,}")
print(f"CNN_batch parameter count: {count_params(model_batch):,}")
print(f"\nCNN architecture:")
print(model_cnn)
print(f"\nCNN_batch architecture:")
print(model_batch)

---
## 3. Training Both Models

In [ ]:
def train_and_evaluate(model, train_loader, test_loader, dataset_val,
                       n_epochs=5, lr=0.1):
    """
    Train model with SGD + Cross Entropy Loss.
    Returns (cost_list, accuracy_list) per epoch.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    N_test    = len(dataset_val)

    cost_list     = []
    accuracy_list = []
    start_time    = time.time()

    for epoch in range(n_epochs):
        model.train()
        cost = 0.0
        for x, y in train_loader:
            optimizer.zero_grad()
            z    = model(x)
            loss = criterion(z, y)
            loss.backward()
            optimizer.step()
            cost += loss.item()

        # Validation
        model.eval()
        correct = 0
        with torch.no_grad():
            for x_test, y_test in test_loader:
                z = model(x_test)
                _, yhat = torch.max(z.data, 1)
                correct += (yhat == y_test).sum().item()

        accuracy = correct / N_test
        cost_list.append(cost)
        accuracy_list.append(accuracy)
        elapsed = time.time() - start_time
        print(f"  Epoch {epoch+1}/{n_epochs} | "
              f"Cost: {cost:.2f} | "
              f"Acc: {accuracy*100:.2f}% | "
              f"Time: {elapsed:.1f}s")

    return cost_list, accuracy_list

In [ ]:
N_EPOCHS = 5
LR = 0.1

# Train CNN
print("Training CNN (standard)...")
torch.manual_seed(0)
model_cnn = CNN(out_1=16, out_2=32, number_of_classes=10)
cost_cnn, acc_cnn = train_and_evaluate(
    model_cnn, train_loader, test_loader, dataset_val,
    n_epochs=N_EPOCHS, lr=LR
)

print()

# Train CNN_batch
print("Training CNN_batch (with Batch Normalization)...")
torch.manual_seed(0)
model_batch = CNN_batch(out_1=16, out_2=32, number_of_classes=10)
cost_batch, acc_batch = train_and_evaluate(
    model_batch, train_loader, test_loader, dataset_val,
    n_epochs=N_EPOCHS, lr=LR
)

---
## 4. Results & Comparison

In [ ]:
# Plot cost and accuracy for both models
epochs = range(1, N_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Cost
axes[0].plot(epochs, cost_cnn,   'b-o',  linewidth=2, markersize=7, label='CNN')
axes[0].plot(epochs, cost_batch, 'r-s',  linewidth=2, markersize=7, label='CNN + BatchNorm')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Training Cost (total loss)', fontsize=12)
axes[0].set_title('Training Cost per Epoch', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=12); axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs, [a*100 for a in acc_cnn],   'b-o',  linewidth=2, markersize=7,
             label=f'CNN (final: {acc_cnn[-1]*100:.2f}%)')
axes[1].plot(epochs, [a*100 for a in acc_batch], 'r-s',  linewidth=2, markersize=7,
             label=f'CNN+BN (final: {acc_batch[-1]*100:.2f}%)')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Validation Accuracy (%)', fontsize=12)
axes[1].set_title('Validation Accuracy per Epoch', fontsize=13, fontweight='bold')
axes[1].set_ylim(0, 100)
axes[1].legend(fontsize=12); axes[1].grid(True, alpha=0.3)

plt.suptitle('Fashion-MNIST: CNN vs CNN with Batch Normalization', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fashion_mnist_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fashion_mnist_results.png")

---
## 5. Error Analysis

In [ ]:
def get_misclassified(model, dataset_val, n=9):
    """Return up to n misclassified (image, true_label, pred_label) tuples."""
    model.eval()
    misclassified = []
    loader = DataLoader(dataset_val, batch_size=256, shuffle=False)
    idx = 0
    with torch.no_grad():
        for x, y in loader:
            z = model(x)
            _, preds = torch.max(z, 1)
            for i in range(len(y)):
                if preds[i].item() != y[i].item():
                    misclassified.append((x[i], y[i].item(), preds[i].item()))
                if len(misclassified) >= n:
                    return misclassified
            idx += len(y)
    return misclassified


# Show misclassified examples from the better model
best_model = model_batch if acc_batch[-1] >= acc_cnn[-1] else model_cnn
best_name  = 'CNN+BatchNorm' if acc_batch[-1] >= acc_cnn[-1] else 'CNN'
errors = get_misclassified(best_model, dataset_val, n=9)

if errors:
    fig, axes = plt.subplots(3, 3, figsize=(9, 9))
    for ax, (img, true_lbl, pred_lbl) in zip(axes.flatten(), errors):
        ax.imshow(img.numpy().squeeze(), cmap='gray')
        ax.set_title(f'True: {CLASS_NAMES[true_lbl]}\nPred: {CLASS_NAMES[pred_lbl]}',
                     fontsize=9, color='red')
        ax.axis('off')
    plt.suptitle(f'Misclassified Samples — {best_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('fashion_mnist_errors.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: fashion_mnist_errors.png")
else:
    print("No misclassified samples found — perfect accuracy!")

In [ ]:
# Per-class accuracy breakdown
def per_class_accuracy(model, dataset_val, n_classes=10):
    """Return per-class accuracy as a dict."""
    model.eval()
    correct = torch.zeros(n_classes)
    total   = torch.zeros(n_classes)
    loader  = DataLoader(dataset_val, batch_size=256, shuffle=False)
    with torch.no_grad():
        for x, y in loader:
            z = model(x)
            _, preds = torch.max(z, 1)
            for c in range(n_classes):
                mask = (y == c)
                correct[c] += (preds[mask] == y[mask]).sum().item()
                total[c]   += mask.sum().item()
    return {CLASS_NAMES[c]: (correct[c] / total[c]).item() * 100 for c in range(n_classes)}


class_acc_cnn   = per_class_accuracy(model_cnn,   dataset_val)
class_acc_batch = per_class_accuracy(model_batch, dataset_val)

print("Per-class accuracy:")
print(f"{'Class':<18} {'CNN':>8} {'CNN+BN':>10}")
print("-" * 40)
for cls in CLASS_NAMES:
    print(f"{cls:<18} {class_acc_cnn[cls]:>7.1f}%  {class_acc_batch[cls]:>7.1f}%")

# Bar chart
x = np.arange(len(CLASS_NAMES))
width = 0.35
fig, ax = plt.subplots(figsize=(13, 5))
bars1 = ax.bar(x - width/2, [class_acc_cnn[c] for c in CLASS_NAMES],
                width, label='CNN', color='steelblue', edgecolor='k')
bars2 = ax.bar(x + width/2, [class_acc_batch[c] for c in CLASS_NAMES],
                width, label='CNN+BatchNorm', color='tomato', edgecolor='k')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=10)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Per-Class Accuracy: CNN vs CNN+BatchNorm', fontsize=13, fontweight='bold')
ax.legend(fontsize=12)
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('fashion_mnist_per_class.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fashion_mnist_per_class.png")

---
## 6. Summary

| Aspect | CNN | CNN + BatchNorm |
|---|---|---|
| Architecture | Conv→ReLU→Pool ×2, FC | Same + BN after each layer |
| Parameters | ~8,330 | ~8,490 (BN adds γ, β) |
| Training stability | Good | Better (reduced covariate shift) |
| Convergence speed | Standard | Often faster |
| Regularisation | None built-in | Mild (BN acts as regulariser) |

### Key Takeaways
1. **Batch Normalization** normalises layer inputs to zero mean and unit variance — this smooths the loss landscape and enables faster learning
2. **Padding=2 with kernel=5** preserves spatial dimensions after convolution (same padding)
3. **CrossEntropyLoss** is the correct loss for multi-class classification — it combines LogSoftmax + NLLLoss
4. Even 5 epochs is sufficient to achieve reasonable accuracy on Fashion-MNIST with a 2-layer CNN

In [ ]:
# Final summary table
print("=" * 55)
print("FINAL RESULTS: Fashion-MNIST CNN Comparison")
print("=" * 55)
print(f"{'Metric':<30} {'CNN':>10} {'CNN+BN':>10}")
print("-" * 55)
print(f"{'Parameters':<30} {count_params(model_cnn):>10,} {count_params(model_batch):>10,}")
print(f"{'Final Val Accuracy':<30} {acc_cnn[-1]*100:>9.2f}% {acc_batch[-1]*100:>9.2f}%")
print(f"{'Final Training Cost':<30} {cost_cnn[-1]:>10.2f} {cost_batch[-1]:>10.2f}")
print(f"{'Epoch 1 Accuracy':<30} {acc_cnn[0]*100:>9.2f}% {acc_batch[0]*100:>9.2f}%")
winner = 'CNN+BatchNorm' if acc_batch[-1] >= acc_cnn[-1] else 'CNN'
print(f"\n✓ Better model: {winner}")